# Research 45 — One-hot and max-bin fixed blend screen

Compare each newly trained candidate standalone and in its predeclared 80/20 blend with the current reference. Include the four-seed leader only as context. Report macro/weighted F1 and seed-42 fold scores; no fitting, weight search, or Test access.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/train.csv").is_file()
)
y = pd.read_csv(ROOT / "data/train.csv", usecols=["segment"]).segment.to_numpy(dtype=int)


def load_probs(filename):
    frame = pd.read_csv(ROOT / "predictions" / filename)
    assert len(frame) == len(y) and np.array_equal(frame.segment.to_numpy(), y), filename
    probs = frame[[f"prob_{i}" for i in range(4)]].to_numpy(dtype=float)
    assert np.isfinite(probs).all() and (probs >= 0).all(), filename
    sums = probs.sum(axis=1, keepdims=True)
    assert (sums > 0).all(), filename
    return probs / sums


p_base = load_probs("segment-oof.csv")
components = {
    "One-hot VIP × region": "segment-lightgbm-onehot-vip-region-oof.csv",
    "max_bin=127 VIP × region": "segment-lightgbm-maxbin127-vip-region-oof.csv",
}
p_leader = load_probs("segment-vip-region-four-seed-average-oof.csv")
folds = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=42).split(np.zeros(len(y)), y))
candidates = {"Current reference": p_base, "Current four-seed leader": p_leader}
blend_outputs = {}
for name, filename in components.items():
    component = load_probs(filename)
    candidates[f"{name}, standalone"] = component
    blend = 0.8 * p_base + 0.2 * component
    candidates[f"{name}, fixed 80/20 blend"] = blend
    blend_outputs[name] = blend
rows = []
for name, probs in candidates.items():
    pred = probs.argmax(axis=1)
    row = {
        "candidate": name,
        "macro_f1": f1_score(y, pred, average="macro"),
        "weighted_f1": f1_score(y, pred, average="weighted"),
    }
    for fold, (_, valid_idx) in enumerate(folds, start=1):
        row[f"fold_{fold}_macro_f1"] = f1_score(y[valid_idx], pred[valid_idx], average="macro")
    rows.append(row)
results = pd.DataFrame(rows)
display(results.round(6))
results.to_csv(ROOT / "assets/segment-onehot-maxbin-blend-screen.csv", index=False)
for name, probs in blend_outputs.items():
    stem = "onehot" if name.startswith("One-hot") else "maxbin127"
    pd.DataFrame(probs, columns=[f"prob_{i}" for i in range(4)]).assign(segment=y).to_csv(
        ROOT / f"predictions/segment-lightgbm-{stem}-vip-region-blend-oof.csv",
        index=False,
    )
print("Fixed candidate-only 80/20 blends evaluated; Test not loaded.")

,candidate,macro_f1,weighted_f1,fold_1_macro_f1,fold_2_macro_f1,fold_3_macro_f1
0,Current reference,0.750895,0.764683,0.749445,0.749821,0.753390
1,Current four-seed leader,0.752172,0.765704,0.750552,0.751365,0.754568
2,"One-hot VIP × region, standalone",0.744308,0.756069,0.742035,0.744263,0.746591
3,"One-hot VIP × region, fixed 80/20 blend",0.751402,0.764894,0.749956,0.750326,0.753895
4,"max_bin=127 VIP × region, standalone",0.741758,0.753995,0.739581,0.740527,0.745138
5,"max_bin=127 VIP × region, fixed 80/20 blend",0.751533,0.765008,0.749944,0.750979,0.753648


Fixed candidate-only 80/20 blends evaluated; Test not loaded.
